# Notebook 1: Preprocessing & Data Loading
### Product Review Rating Prediction — Amazon Reviews (5-class)

This notebook handles:
- Loading the SetFit/amazon_reviews_multi_en dataset
- Classic NLP preprocessing for the BiGRU baseline (vocabulary building, fastText embeddings)
- RoBERTa tokenization pipeline
- Saving processed data for downstream notebooks

In [ ]:
# ── Install dependencies (Kaggle) ──────────────────────────────────────────────
!pip install -q datasets transformers[torch] gdown fasttext-wheel
# If fasttext-wheel fails, use the gensim fasttext loader instead
!pip install -q gensim spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, re, pickle, random, collections
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import RobertaTokenizerFast
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Output directory
os.makedirs('processed', exist_ok=True)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Load Dataset

In [ ]:
print('Loading dataset...')
dataset = load_dataset('SetFit/amazon_reviews_multi_en')
print(dataset)

train_ds = dataset['train']
val_ds   = dataset['validation']
test_ds  = dataset['test']

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print('Example:', train_ds[0])

In [ ]:
# Inspect label distribution
import matplotlib.pyplot as plt

labels = train_ds['label']
counts = collections.Counter(labels)
print('Label distribution (train):', dict(sorted(counts.items())))

plt.bar([str(k+1) for k in sorted(counts)], [counts[k] for k in sorted(counts)])
plt.xlabel('Star Rating'); plt.ylabel('Count'); plt.title('Training Label Distribution')
plt.tight_layout(); plt.savefig('processed/label_dist.png', dpi=150); plt.show()

## 2. Classic NLP Preprocessing (Baseline BiGRU)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

HTML_RE  = re.compile(r'<[^>]+>')
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
MULTI_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    """Lowercase, strip HTML tags, URLs, and extra whitespace."""
    text = HTML_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = text.lower()
    text = MULTI_RE.sub(' ', text).strip()
    return text

def tokenize(text: str):
    """SpaCy tokenization, returning list of token strings."""
    doc = nlp(text)
    return [tok.text for tok in doc if not tok.is_space]

def get_text(sample):
    """Concatenate review_title and review_body."""
    if sample.get('text'):   # SetFit dataset: title + body already merged in 'text'
        return sample['text'].strip()
    title = sample.get('review_title', '') or ''
    body  = sample.get('review_body', '')  or ''
    return (title + ' ' + body).strip()

# Quick test
ex = train_ds[0]
raw = get_text(ex)
print('Raw:', raw[:200])
print('Cleaned:', clean_text(raw)[:200])
print('Tokens:', tokenize(clean_text(raw))[:20])

In [ ]:
MAX_SEQ_LEN  = 200
MIN_FREQ     = 3
PAD_TOKEN    = '<PAD>'
UNK_TOKEN    = '<UNK>'

# ── Build vocabulary from training set ────────────────────────────────────────
print('Building vocabulary...')
word_freq = collections.Counter()

train_texts_raw = []
train_labels    = []

for sample in tqdm(train_ds, desc='Tokenizing train'):
    text   = clean_text(get_text(sample))
    tokens = tokenize(text)
    word_freq.update(tokens)
    train_texts_raw.append(tokens)
    train_labels.append(sample['label'])

vocab = [PAD_TOKEN, UNK_TOKEN] + [
    w for w, c in word_freq.items() if c >= MIN_FREQ
]
word2idx = {w: i for i, w in enumerate(vocab)}
print(f'Vocabulary size: {len(vocab):,}')

pickle.dump(word2idx, open('processed/word2idx.pkl', 'wb'))

In [ ]:
def encode_tokens(tokens, word2idx, max_len=MAX_SEQ_LEN):
    """Convert tokens to padded integer IDs."""
    unk_id = word2idx[UNK_TOKEN]
    ids = [word2idx.get(t, unk_id) for t in tokens[:max_len]]
    pad_len = max_len - len(ids)
    ids += [0] * pad_len  # PAD_TOKEN is index 0
    return ids

# Encode training set
train_ids = np.array(
    [encode_tokens(toks, word2idx) for toks in tqdm(train_texts_raw, desc='Encoding train')],
    dtype=np.int32
)

# Encode val / test
def process_split(split_ds):
    ids_list, labels_list = [], []
    for sample in tqdm(split_ds):
        text   = clean_text(get_text(sample))
        tokens = tokenize(text)
        ids_list.append(encode_tokens(tokens, word2idx))
        labels_list.append(sample['label'])
    return np.array(ids_list, dtype=np.int32), np.array(labels_list, dtype=np.int64)

val_ids,  val_labels  = process_split(val_ds)
test_ids, test_labels = process_split(test_ds)
train_labels_arr = np.array(train_labels, dtype=np.int64)

np.save('processed/train_ids.npy',    train_ids)
np.save('processed/train_labels.npy', train_labels_arr)
np.save('processed/val_ids.npy',      val_ids)
np.save('processed/val_labels.npy',   val_labels)
np.save('processed/test_ids.npy',     test_ids)
np.save('processed/test_labels.npy',  test_labels)
print('Classic NLP arrays saved.')

## 3. FastText / GloVe Embedding Matrix

In [ ]:
# ── FastText Pre-trained Word Vectors ─────────────────────────────────────────
# NOTE: SetFit/amazon_reviews_multi_en is our REVIEW DATASET (loaded in section 1).
# fastText vectors are SEPARATE — they are pre-trained 300-dim word embeddings
# used only to initialize the BiGRU embedding layer, as required by the task.
# Task spec says: 'pre-trained fastText (or GloVe) word embeddings as its input layer'

EMBED_DIM        = 300
FASTTEXT_VEC_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip'
FASTTEXT_LOCAL   = 'crawl-300d-2M.vec'
KAGGLE_FT_PATH   = '/kaggle/input/fasttext-crawl-300d-2m/crawl-300d-2M.vec'

# Priority 1: Pre-attached Kaggle dataset (no download needed)
# Priority 2: Auto-download directly from Meta AI fastText servers
embed_path = None
if os.path.exists(KAGGLE_FT_PATH):
    embed_path = KAGGLE_FT_PATH
    print(f'Using Kaggle dataset input: {embed_path}')
elif os.path.exists(FASTTEXT_LOCAL):
    embed_path = FASTTEXT_LOCAL
    print(f'Using cached local file: {embed_path}')
else:
    print('Downloading fastText vectors from Meta AI (~1.6 GB zip)...')
    import urllib.request, zipfile
    urllib.request.urlretrieve(FASTTEXT_VEC_URL, 'crawl-300d-2M.vec.zip')
    with zipfile.ZipFile('crawl-300d-2M.vec.zip', 'r') as z:
        z.extractall('.')
    os.remove('crawl-300d-2M.vec.zip')
    embed_path = FASTTEXT_LOCAL
    print(f'Done. Extracted: {embed_path}')

# ── Build embedding matrix ─────────────────────────────────────────────────────
print(f'Building embedding matrix from {embed_path} ...')
embedding_matrix = np.random.normal(0, 0.1, (len(vocab), EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = 0.0  # PAD token => zero vector
hit, total = 0, len(vocab)
with open(embed_path, encoding='utf-8', errors='ignore') as f:
    for line in tqdm(f, desc='Loading fastText vectors'):
        parts = line.rstrip().split(' ')
        if len(parts) != EMBED_DIM + 1:
            continue  # skip header line if present
        word = parts[0]
        if word in word2idx:
            embedding_matrix[word2idx[word]] = np.array(parts[1:], dtype=np.float32)
            hit += 1
print(f'Coverage: {hit}/{total} vocab words = {hit/total*100:.1f}%')

np.save('processed/embedding_matrix.npy', embedding_matrix)
print(f'Embedding matrix saved. Shape: {embedding_matrix.shape}')

## 4. RoBERTa Tokenization

In [ ]:
ROBERTA_MAX_LEN = 256
ROBERTA_MODEL   = 'roberta-base'

tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL)

def tokenize_roberta(examples):
    texts = [t or '' for t in examples['text']]
    return tokenizer(
        texts,
        max_length=ROBERTA_MAX_LEN,
        truncation=True,
        padding=False,      # dynamic padding via DataCollator at training time
    )

print('Tokenizing for RoBERTa...')
# We keep the HuggingFace dataset format and save as arrow/disk
tokenized_train = train_ds.map(tokenize_roberta, batched=True, batch_size=1000,
                                remove_columns=[c for c in train_ds.column_names if c != 'label'],
                                desc='Tokenizing train')
tokenized_val   = val_ds.map(tokenize_roberta,   batched=True, batch_size=1000,
                              remove_columns=[c for c in val_ds.column_names if c != 'label'],
                              desc='Tokenizing val')
tokenized_test  = test_ds.map(tokenize_roberta,  batched=True, batch_size=1000,
                               remove_columns=[c for c in test_ds.column_names if c != 'label'],
                               desc='Tokenizing test')

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')
tokenized_test.set_format('torch')

tokenized_train.save_to_disk('processed/roberta_train')
tokenized_val.save_to_disk('processed/roberta_val')
tokenized_test.save_to_disk('processed/roberta_test')
print('RoBERTa tokenized datasets saved to disk.')

In [ ]:
# Also tokenize a 128-token version for the ablation study
def tokenize_roberta_128(examples):
    texts = [t or '' for t in examples['text']]
    return tokenizer(texts, max_length=128, truncation=True, padding=False)

tokenized_train_128 = train_ds.map(tokenize_roberta_128, batched=True, batch_size=1000,
                                    remove_columns=[c for c in train_ds.column_names if c != 'label'],
                                    desc='Tokenizing train (128)')
tokenized_val_128   = val_ds.map(tokenize_roberta_128,   batched=True, batch_size=1000,
                                  remove_columns=[c for c in val_ds.column_names if c != 'label'],
                                  desc='Tokenizing val (128)')
tokenized_test_128  = test_ds.map(tokenize_roberta_128,  batched=True, batch_size=1000,
                                   remove_columns=[c for c in test_ds.column_names if c != 'label'],
                                   desc='Tokenizing test (128)')

tokenized_train_128.set_format('torch')
tokenized_val_128.set_format('torch')
tokenized_test_128.set_format('torch')

tokenized_train_128.save_to_disk('processed/roberta_train_128')
tokenized_val_128.save_to_disk('processed/roberta_val_128')
tokenized_test_128.save_to_disk('processed/roberta_test_128')
print('128-token datasets saved.')

In [ ]:
print('\n=== Preprocessing Summary ===')
print(f'Vocab size (classic): {len(vocab):,}')
print(f'Train IDs shape:      {train_ids.shape}')
print(f'Embedding matrix:     {embedding_matrix.shape}')
print(f'RoBERTa max len:      {ROBERTA_MAX_LEN}')
print(f'RoBERTa tokenized:    train={len(tokenized_train)}, val={len(tokenized_val)}, test={len(tokenized_test)}')
print('All preprocessing complete.')